# Capstone: Study Tip Generator (GPT-2 fine-tune)
**Course:** ETMMGA304 - Generative AI and Its Applications (MCA(AI/ML))


**Name**:- Kulsum Banoo,
**Roll_No**:-2501940072,
**Date**:-20-11-2026

**Aim:** Give beginners one actionable study tip for a topic; fine-tune the last GPT-2 block and compare base vs adapted on held-out topics.

**Software:** Google Colab, PyTorch, Matplotlib | **Seed:** 42

In [ ]:
from pathlib import Path
import random
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out = Path("lab_outputs")
out.mkdir(exist_ok=True)
print("Device:", device)

Device: cuda


In [ ]:
%pip install -q "transformers>=4.51,<5" pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

train_pairs = [
    ("graphs", "Sketch a graph, then mark a path through three vertices."),
    ("Python loops", "Trace a loop by writing the counter after each step."),
    ("fractions", "Draw equal parts and label the fraction you shaded."),
    ("sets", "Draw a Venn diagram and label the intersection."),
    ("sorting", "Sort five values by hand and record each swap."),
    ("probability", "List the outcomes of two coin tosses before counting."),
    ("functions", "Write the input and output of a function in a two-column table."),
    ("stacks", "Push and pop five cards by hand to see last in, first out."),
    ("queues", "Line up five items and remove them from the front one by one."),
    ("linked lists", "Draw boxes and arrows, then redraw them after inserting a node."),
    ("trees", "Draw a small tree and label the root, parent and children."),
    ("SQL joins", "Write two tiny tables and match their rows by hand first."),
    ("variables", "Write a variable name on a card and change the value inside."),
    ("conditionals", "List three inputs and trace which branch each one takes."),
    ("algebra", "Solve one step at a time and check by substituting the answer."),
    ("statistics", "Calculate the mean of five numbers by hand before using code."),
]
# Held-out topics (NOT used in training)
held_out = ["recursion", "matrices", "binary search", "hash tables", "pointers",
            "dynamic programming", "graph traversal", "normalization", "regular expressions", "big-O notation"]
print(len(train_pairs), "train pairs |", len(held_out), "held-out topics")

16 train pairs | 10 held-out topics


In [ ]:
model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(model_id).to(device).eval()

tuned = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tuned.requires_grad_(False)
tuned.transformer.h[-1].requires_grad_(True)
opt = torch.optim.AdamW(tuned.transformer.h[-1].parameters(), lr=5e-5)

EPOCHS = 5
for epoch in range(EPOCHS):
    tuned.train()
    random.shuffle(train_pairs)
    total = 0.0
    for topic, answer in train_pairs:
        prefix = tokenizer.encode(f"Topic: {topic}\nStudy tip:")
        target = tokenizer.encode(" " + answer) + [tokenizer.eos_token_id]
        ids = torch.tensor([prefix + target], device=device)
        labels = torch.tensor([[-100] * len(prefix) + target], device=device)
        opt.zero_grad()
        loss = tuned(input_ids=ids, labels=labels).loss
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1} mean loss: {total/len(train_pairs):.3f}")
tuned.eval()
tuned.save_pretrained(out / "study_tip_gpt2")
tokenizer.save_pretrained(out / "study_tip_gpt2")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1 mean loss: 4.947
Epoch 2 mean loss: 4.588
Epoch 3 mean loss: 4.476
Epoch 4 mean loss: 4.174
Epoch 5 mean loss: 3.774


('lab_outputs/study_tip_gpt2/tokenizer_config.json',
 'lab_outputs/study_tip_gpt2/special_tokens_map.json',
 'lab_outputs/study_tip_gpt2/vocab.json',
 'lab_outputs/study_tip_gpt2/merges.txt',
 'lab_outputs/study_tip_gpt2/added_tokens.json',
 'lab_outputs/study_tip_gpt2/tokenizer.json')

In [ ]:
def gen(m, topic):
    inputs = tokenizer(f"Topic: {topic}\nStudy tip:", return_tensors="pt").to(device)
    with torch.no_grad():
        o = m.generate(**inputs, do_sample=False, max_new_tokens=45,
                       pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(o[0, inputs.input_ids.size(1):], skip_special_tokens=True).strip()

rows = []
for t in held_out:
    for name, m in (("Base", base), ("Adapted", tuned)):
        rows.append({"Topic": t, "Model": name, "Output": gen(m, t),
                     "Relevance": None, "Readability": None, "Usefulness": None, "Notes": ""})
df = pd.DataFrame(rows)
df.to_csv(out / "evaluation_raw.csv", index=False)
df

,Topic,Model,Output,Relevance,Readability,Usefulness,Notes
0,recursion,Base,Use the recursion keyword to define a new type...,None,None,None,
1,recursion,Adapted,Use the recursion keyword to define a new vari...,None,None,None,
2,matrices,Base,Use the matrices to create a matrix of the sam...,None,None,None,
3,matrices,Adapted,Use the matrix as a starting point.,None,None,None,
4,binary search,Base,Use the search box to find the search term you...,None,None,None,
5,binary search,Adapted,Use the search box at the top of the page to f...,None,None,None,
6,hash tables,Base,Use the hash table to find the most common has...,None,None,None,
7,hash tables,Adapted,Use the hash table to find the most common table.,None,None,None,
8,pointers,Base,Use the following to get the most out of your ...,None,None,None,
9,pointers,Adapted,Use the pointer to store the value of a pointe...,None,None,None,


In [ ]:
import shutil
from google.colab import files
shutil.make_archive("study_tip_gpt2", "zip", out / "study_tip_gpt2")
files.download("study_tip_gpt2.zip")
files.download(str(out / "evaluation_raw.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>